# Meta-fairness-scores with FairBench
FairBench computes a wide range of fairness measures in its reports. Each of those already contains hundreds of considerations. However, it is often the case that further simplification is necessary to obtain a very high-level aggregate picture before even looking at surface issues (which would then lead to investigating numberical details).

For this reason, the ability to post-process reports with simplification mechanisms is provided. This works like other filters. There are three broad types of post-processors:
a) those that perform adjustments so that bias or fairness measure values better reflect real-world properties (e.g., keeping only bias measures, adjusting thresholds with intuition driven from basic fuzzy logic, etc.)
b) those that summarize specific collections of concerns for a specific subset of assessment (e.g., common fairness stamps, compliance with legal bodies, compliance with specific standards)
c) as a category that partially overlaps and is used by the above, those that reduce fairness assessments to one coherent value that can nonentheless be further explored.

**Draft status:** This notebook currently contains implementations that generalize ideas presented by the independent FairBench-genai project here: https://prasannavijay.github.io/fairbench/whitepaper/

## Multilabel classification
First import the library and create a fairness report for multilabel classification.

In [1]:
import fairbench as fb

x, y, yhat = fb.bench.tabular.compas(test_size=0.2)
sensitive = fb.Dimensions(fb.categories @ x["sex"], fb.categories @ x["race"])
sensitive = sensitive.intersectional().strict()
multireport = fb.reports.pairwise(multipredictions=yhat, multilabels=y, sensitive=sensitive)

We will also need a supossedely more biased baseline report (less training data; data volume does not necessarily impact fairness, but this is for demonstration only) to check for bias amplifications later.

In [2]:
baseline_x, baseline_y, baseline_yhat = fb.bench.tabular.compas(test_size=0.5)
baseline_sensitive = fb.Dimensions(fb.categories @ baseline_x["sex"], fb.categories @ baseline_x["race"])
baseline_sensitive = baseline_sensitive.intersectional().strict()
baseline_report = fb.reports.pairwise(multipredictions=baseline_yhat, multilabels=baseline_y, sensitive=baseline_sensitive)

First, **output diversity entropy (ode)** is now part of multilabel reports. (By the way, check out that Html visualizations are now directly embedded in Jupyter. 

In [3]:
multireport.ode.wmean.show(env=fb.export.Html)

Now we will compare the report with its baseline to obtain the **bias amplification** for all computed measures. This can also apply to any other report, such as for binary classification, regression, `.vsall` instead of `.pairwise`, etc...

Bias amplification checks can be performed by filtering with the namesake investigator. Note that we first chain another investigator that checks the amplification of bias measures only (with ideal target value 0, conversely fairness measures typically have ideal target 1). Using `force=True` to avoid the need for filtering the `baseline_report`; this would have been needed for exact comparison compatibility otherwise.

In [4]:
amplification = multireport.filter(fb.investigate.IsBias, fb.investigate.Amplification(base=baseline_report, force=True))
amplification["comparison of maxerror"].show()  # see other specializations and details with amplification.help()


##### maxerror comparison #####
|This reduction is comparison of the maximum deviation from the ideal value.
|Computations cover several cases.

  (0.0, 0.3521126760563376)
  ▎    █           █      
  ▎ ▆  █  ▆        █  ▆   
  ▎ █  █  █  ▄  █  █  █  ▆
  ▎ █  █  █  █  █  █  █  █
  ▎ █  █  █  █  █  █  █  █
  ▎▬*▬▬-▬▬+▬▬x▬▬o▬▬□▬▬◇▬▬#
                (8.0, 0.0)
  
   * wmacc comparison                    0.290 maxerror wmacc/maxerror wmacc
   - amacc comparison                    0.352 maxerror amacc/maxerror amacc
   + gmacc comparison                    0.340 maxerror gmacc/maxerror gmacc
   x amppv comparison                    0.272 maxerror amppv/maxerror amppv
   o gmppv comparison                    0.248 maxerror gmppv/maxerror gmppv
   □ amtpr comparison                    0.352 maxerror amtpr/maxerror amtpr
   ◇ gmtpr comparison                    0.340 maxerror gmtpr/maxerror gmtpr
   # ode comparison                      0.221 maxerror ode/maxerror ode





## Binary report
We will continue with a simple binary report, as positive rate computations needed for one of the next examples apply only to binary classification (or multiclass classification converted to a binary setting by considering a one-vs-all scenario for the "positive" class). For conciseness, visualization focuses on the positive rates below and uses the minimal `ToString` export mechanism that tries to be a short as possible (try `Console` or `Html` for more details).

In [5]:
report = fb.reports.pairwise(predictions=yhat, labels=y, sensitive=sensitive)
print(report.maxrel.pr.show(env=fb.export.ToString))

[measure] pr                             0.599 maxrel pr (ideal value 0.000, abs bound 1.000)


Another relevant concept is the ability to obtain the **worst-case bias**, as in, the worst deviation of fairness or bias measures from target values. This is just another filtering on reports that drastically simplifies the result but may be impossible to fully optimize (see caveats and recommendations). Again, this applies to any report.

In [6]:
print(report.filter(fb.investigate.WorstCase).show(fb.export.ToString))

[summary] worst bias                     2.597 worst bias (ideal value 0.000)


We will be computing the **representation skew index** (rsi) by comparing the positive rates with a desired distribution obtained from sensitive attribute dimension representations. To this end, we first get the mean on each dimension and convert the result to a dummy report for comparison.

In [7]:
target_pr_values = fb.core.report_from_dims(sensitive.sum()/sensitive.shape[0])
print(target_pr_values.show(env=fb.export.ToString, depth=2)) # this remains a concise way to show reports

[analysis] multidim                     
  [group] Female&African-American        0.096 
  [group] Male&Other                     0.046 
  [group] Male&African-American          0.440 
  [group] Female&Other                   0.009 
  [group] Female&Caucasian               0.073 
  [group] Female&Hispanic                0.011 
  [group] Male&Caucasian                 0.242 
  [group] Male&Hispanic                  0.079 


We can now focus on the positive rates for all sensitive demensions of the report, obtained via `report.min.pr.details`. Dunno worry if you have trouble analytically thinking of this pattern; this is its main use case you will use 99% of the time (with or without `details`, which here basically strip away the minimum positive rate value to keep only its dependent computations). 

We will replace the report's ideal target values via another investigator's filtering, and finally compute the `SkewIndex` as the KL divergence between normalized desired values and normalized computed values. This is done across all report values, but rememember that now we are focusing on a sub-report that just contains the the positive rates of all groups.

In [8]:
prreport = report.min.pr.details.filter(fb.investigate.SetTargets(targets=target_pr_values), fb.investigate.SkewIndex)
prreport.show(env=fb.export.Html)

Do note that the base report to be compared against **can capture manually inputted values** too. Below is a simple example.

In [9]:
import fairbench as fb

manual_pr_values = fb.core.report_from_dims({"male": 0.3, "female": 0.4})
print(manual_pr_values)
manual_y = [1,0,1,1,1,0]
manual_yhat  = [1,0,1,0,1,0]
manual_sensitive = fb.Dimensions(fb.categories@["male", "male", "male", "female", "female", "female"])
manual_report = fb.reports.pairwise(labels=manual_y,predictions=manual_yhat,sensitive=manual_sensitive)
print(manual_report.min.pr.details)
manual_report.min.pr.details.filter(fb.investigate.Amplification(base=manual_pr_values, default_to_zero_targets=True)).show()

[analysis] multidim                     
  [group] male                           0.300 
  [group] female                         0.400 
[measure view] pr groups                
  [group] female                         0.333 pr
  [group] male                           0.667 pr

##### pr groups comparison #####
|This measure of a view is comparison of the minimum of the positive rate 
|across groups.
|Computations cover several cases.

  (0.0, 2.2222222222222223)
  ▎    █
  ▎    █
  ▎    █
  ▎    █
  ▎ █  █
  ▎▬*▬▬-
  (2.0, 0.0)
  
   * female comparison                   0.833 female/female
   - male comparison                     2 male/male



